# 第1回：ガイダンス：組合せ最適化とブラックボックス最適化

```{dropdown} NOTE: この資料について
:class: dropdown
:open: true
この資料は，この授業全体のガイダンスとして，組合せ最適化とブラックボックス最適化の位置づけをまとめた学習用テキストである。  
図や式は補足説明とあわせて読むと理解しやすい。
```

---


## 0. この授業で何を学ぶか

本授業では、 **「多くの候補の中から、評価値が良く、必要なら制約も満たす解を見つける方法」** を学ぶ。  
前半では順列や 0/1 ベクトルのような **組合せ最適化** を扱い，後半では目的関数の中身を直接使わず，入力に対する評価値だけを頼りに探索する **ブラックボックス最適化** を扱う。

扱うアルゴリズムは，バイナリ遺伝的アルゴリズム（Binary GA），遺伝的アルゴリズム（GA），粒子群最適化（PSO），アリコロニー最適化（ACO），差分進化（DE），CMA-ES である。いずれも「候補解を作る」「評価する」「良かった情報を次に使う」という共通の見方で読める。


### この科目の到達目標（再掲）
- 組合せ最適化とブラックボックス最適化の基本的な考え方を理解し、実社会の問題に適用できる。


### 第1回の到達目標
第1回終了時に、次を説明できる状態を目指す。

1. 「最適化問題」「組合せ最適化問題」「ブラックボックス最適化問題」「制約」の違い  
2. なぜ全探索が難しいのか（組合せ爆発）  
3. 典型的な組合せ最適化問題とブラックボックス最適化問題の例  
4. 厳密解法，近似解法，メタヒューリスティクス，評価回数予算の立ち位置  
5. 本授業で扱う内容と評価方法

---


## 1. 最適化問題の基本


### 1.1 最適化問題とは

最適化問題は、一般に次の形で記述できる。

$$
\text{minimize (or maximize)} \quad f(x) \\
\text{subject to} \quad x \in \mathcal{X}, \ g_i(x)\le 0,\ h_j(x)=0
$$

- $x$: 解（意思決定変数）  
- $f(x)$: 目的関数（評価指標）  
- $g_i, h_j$: 制約条件  
- $\mathcal{X}$: 解候補の集合

ここで重要なのは，$x$ が必ずしも連続値とは限らないことである。$x$ は都市の並び順でも，品物を選ぶ 0/1 ベクトルでも，機械学習モデルのハイパーパラメータでもよい。


```{dropdown} NOTE: 数式表現の読み方
:class: dropdown
:open: true
ここでの英語表現は次の意味です。

- `minimize (or maximize)`: 「目的関数 $f(x)$ をできるだけ小さく（または大きく）する」という意味です。  
- `subject to`: 「次の条件を満たしながら」という意味で、後ろに続くものが制約条件です。  

したがって、式全体は  
「制約 $x \in \mathcal{X},\ g_i(x)\le0,\ h_j(x)=0$ を満たす $x$ の中で、$f(x)$ をできるだけ小さく（または大きく）する」  
という意味になります。

また、似た記号として `argmin` / `argmax` がありますが、役割が少し異なります。

- `minimize f(x)`: 「$f(x)$ を小さくしたい」という **問題の型（目標）** を書いています。  
- $x^{*} = \operatorname{argmin}\, f(x)$: 「$f(x)$ を最小にする **解そのもの $x^{*}$**」を表しています。  

この資料の式は「どんな問題を解きたいか」を宣言しているので `minimize` / `subject to` を使っており、後で「見つかった最適解」を記号で書くときに `argmin` / `argmax` を使う、という使い分けをします。
```


```{important}
:class: dropdown
:open: true
「良い解」とは、目的関数の値が良いだけでなく、制約を満たしている解である。制約違反の解は通常「実行不能解（feasible でない解）」として扱う。
```


### 1.2 組合せ最適化とは

組合せ最適化は、解が **離散的な組合せ** で表される最適化問題である。  
例として次がある。

- 都市の訪問順序（TSP: Traveling Salesman Problem）  
- 品物選択（KP: Knapsack Problem）  
- 作業の割当（AP: Assignment Problem）  
- 時間割の作成（Timetabling Problem）

- 変数が連続値ではなく、離散値（0/1、順列、集合選択）  
- 解空間が有限だが、規模が大きいと現実的に全探索できない  
- 制約が複雑になりやすい（実務要件を反映しやすい）


### 1.3 ブラックボックス最適化とは

ブラックボックス最適化は，目的関数 $f$ の中身を直接利用せず，**候補 $x$ を入れると評価値 $f(x)$ が返ってくる** という前提で良い $x$ を探す最適化である。

たとえば次のような状況である。

- あるハイパーパラメータで機械学習モデルを学習し，交差検証精度を測る  
- ある設定でシミュレーションを 1 回走らせ，コストや性能値を得る  
- 製造条件を変えて実験し，品質スコアを測る  
- ゲーム AI や制御器のパラメータを変えて，試行結果のスコアを見る

この場合，$f(x)$ の式を微分したり，内部構造を解析したりできないことが多い。したがって，「どこを次に試すか」をうまく決めることが重要になる。評価 1 回が重い場合には，**評価回数の予算** も最適化設計の一部である。

```{dropdown} NOTE: 組合せ最適化とブラックボックス最適化の関係
:class: dropdown
:open: true
この 2 つは排他的な分類ではない。解が順列や 0/1 ベクトルなら組合せ最適化であり，その目的関数の中身を使えず評価値だけで探索するならブラックボックス最適化でもある。  
一方，CMA-ES のように主に連続値ベクトルを扱う手法は，組合せ最適化というより連続値のブラックボックス最適化として理解するほうが自然である。
```


### 1.4 組合せ爆発と評価回数の壁

選択肢の数は、要素数 $n$ の増加に対して指数関数的・階乗的に増える場合が多い。

- 0/1選択問題: 候補数 $2^n$  
- 順列問題: 候補数 $n!$

例：20都市の巡回順序（始点固定）  
$$
(20-1)! = 19! \approx 1.216\times 10^{17}
$$

1秒あたり1億通りを評価できても、全探索には現実的でない時間が必要になる。

ブラックボックス最適化でも似た困難がある。候補が連続値なら候補数は有限個として数えにくいが，評価 1 回に学習・実験・シミュレーションが必要なら，何千回も無制限には試せない。つまり，組合せ最適化では **候補数の爆発** が，ブラックボックス最適化では **評価回数の制約** が大きな壁になる。

---


## 2. 典型問題のイメージ

この授業では，解の表現が「順列」「0/1 ベクトル」「連続ベクトル」のどれに近いかを意識する。解の表現が変わると，使いやすいアルゴリズムも変わる。


### 2.1 巡回セールスマン問題（TSP）

- 複数都市を1回ずつ訪問して出発点に戻る  
- 総移動距離（または時間）を最小化  
- 解は都市の順列で表される

応用例:
- 配送ルート最適化  
- 点検巡回計画  
- 基板穴あけ順序最適化


### 2.2 ナップサック問題

- 重さ制限以内で価値が最大となる品物集合を選ぶ  
- 0/1の選択変数で表現しやすい  
- 多くの実務問題の簡略モデルになる

応用例:
- 限られた予算での施策選択  
- 計算資源制限下での機能選択  
- 在庫制約下での積載計画


### 2.3 割当問題・スケジューリング

- 人員や機械をタスクへ割り当てる  
- 納期、優先度、能力差、連続作業制約などが関与  
- 現場の要求を制約としてモデル化しやすい

応用例:
- 時間割編成  
- シフト作成  
- 生産計画


### 2.4 ハイパーパラメータ探索

- 機械学習モデルの設定値を変えながら検証精度を最大化する  
- 例: SVM の $C$ や $\gamma$，ランダムフォレストの木の数，ニューラルネットワークの学習率  
- 目的関数は「学習して検証する」処理全体になるため，中身を微分して使うことは難しい

応用例:
- 限られた学習時間で精度のよい設定を探す  
- シミュレーションや実験の条件を調整する  
- 連続値パラメータを持つ制御器をチューニングする


### 2.5 この授業での対応関係

| 回 | 主な手法 | 典型的な解の表現 | 主な見方 |
| --- | --- | --- | --- |
| 第2回 | Binary GA | 0/1 ベクトル | 特徴を使う／使わないを探索する |
| 第3回 | GA | 順列 | 巡回順序のような組合せを探索する |
| 第4回 | PSO | 連続ベクトル，0/1 ベクトル | 粒子の位置と速度を使って探索する |
| 第5回 | ACO | グラフ上の経路 | 辺に蓄積する情報を使って順路を作る |
| 第6回 | DE | 連続ベクトル | 個体差分から新しい候補を作る |
| 第7〜8回 | CMA-ES | 連続ベクトル | 探索分布の形を学習する |

```{tip}
:class: dropdown
:open: true
現実問題では「単一の目的関数」だけでなく、複数目的（コスト・品質・公平性など）のバランスを取る必要がある。  
このときは重み付き和や制約化などで単一目的に落とし込むことが多い。ブラックボックス最適化でも同じで，たとえば「精度は高く，計算時間は短く」という複数の要求を 1 つの評価値へまとめる設計が必要になる。
```

---


## 3. どう解くか：解法の全体像


### 3.1 厳密解法と近似解法


### 厳密解法
- 最適解を保証する  
- 問題規模が大きいと計算時間が急増しやすい  
- 例: 分枝限定法、動的計画法、整数計画ソルバ  
- 組合せ最適化では重要だが，本授業では主に比較対象として位置づける


### 近似解法・ヒューリスティクス
- 計算時間を実用的に抑え、良い解を得る  
- 最適性保証は弱くなる（またはない）  
- 現場では「短時間で十分良い解」が重要になる場面が多い  
- ブラックボックス最適化では，評価回数を抑えながら良い候補へ近づくことが重要である


### 3.2 メタヒューリスティクス

メタヒューリスティクスは、特定問題に依存しない汎用的な探索枠組みである。

代表例:
- 遺伝的アルゴリズム（GA）: 個体集団に選択・交叉・突然変異を適用する  
- 粒子群最適化（PSO）: 粒子の位置と速度を更新する  
- アリコロニー最適化（ACO）: グラフの辺に蓄積する情報を使う  
- 差分進化（DE）: 個体同士の差分ベクトルから試行点を作る  
- CMA-ES: 多変量正規分布の平均・分散・向きを更新する

```{warning}
:class: dropdown
:open: true
メタヒューリスティクスは「万能アルゴリズム」ではない。  
解表現、近傍、制約処理、パラメータ設定，評価回数の予算が性能を大きく左右する。
```

---


## 4. 産業・社会での活用例

1. 物流最適化  
   - 配送順序、積載、車両台数、時間帯制約を考慮して総コストを削減する。

2. 製造計画  
   - 段取り替え時間・機械能力・納期を考慮し、遅延と在庫を低減する。製造条件のチューニングはブラックボックス最適化として扱えることもある。

3. 情報システム運用  
   - サーバ資源割当、ジョブスケジューリング、ネットワーク経路選択を最適化する。性能測定を繰り返して設定値を調整する問題もある。

4. 教育現場  
   - 時間割編成、試験監督割当、教室利用計画を制約付きで調整する。

5. AIモデル設計  
   - ハイパーパラメータ探索、NAS、進化的探索により設計を効率化する。学習と検証を 1 回の評価として見れば，典型的なブラックボックス最適化である。

---


## 5. この授業の進め方（第1回時点）


### 5.1 週ごとの流れ（概要）

- 第1回: ガイダンス：組合せ最適化とブラックボックス最適化  
- 第2回: バイナリ遺伝的アルゴリズム（Binary GA）  
- 第3回: 遺伝的アルゴリズム（GA）  
- 第4回: 粒子群最適化（PSO）  
- 第5回: アリコロニー最適化（ACO）  
- 第6回: 差分進化（DE）  
- 第7〜8回: 共分散行列適応進化戦略（CMA-ES）  
- 以降: 演習，応用問題，発表・まとめ


### 5.2 成績評価

- 課題: 60%  
- レポート: 30%  
- 発表: 10%  
- 合計60点以上で単位修得

評価観点:
- 理論理解 30%  
- 実装能力 30%  
- プロジェクト演習 40%


### 5.3 提出物の基本方針

- 使用言語は Python / R / Julia のいずれか  
- 再現可能な形で提出（実行手順・依存関係・入力データを明記）  
- 「動いた」だけでなく、設計意図・評価指標・結果解釈を示す  
- ブラックボックス最適化では，評価回数，乱数シード，探索範囲も明記する

---


## 6. 第1回ミニ演習（授業内）

```{admonition} 演習テーマ
:class: seealso dropdown
:open: true
「自分の身近な問題を1つ、組合せ最適化またはブラックボックス最適化として定義する」
```


### 手順
1. 問題を1つ選ぶ（例: 時間割調整、課題実施順、移動ルート、モデル設定の調整）  
2. 目的関数を1つ決める（例: 時間最小、移動距離最小、満足度最大、検証精度最大）  
3. 制約を3つ以上書く（例: 時間上限、優先科目、連続作業禁止、計算時間上限）  
4. 解の表現を定義する（例: 順列、0/1ベクトル、割当表、連続値ベクトル）  
5. その問題が組合せ最適化，ブラックボックス最適化，またはその両方のどれに近いかを書く  
6. 全探索が難しい理由，または評価回数を無制限に増やせない理由を書く


### 提出フォーマット（授業内メモ）

以下のテンプレートを利用すること。

```text
[問題名]

1. 目的関数:
2. 制約:
   - C1:
   - C2:
   - C3:
3. 解の表現:
4. 問題の型:
   - 組合せ最適化 / ブラックボックス最適化 / 両方
5. 解候補数または探索範囲:
6. 全探索または評価回数の面で難しい理由:
```

---


## 7. 実装に向けた準備（次回まで）

次回以降は簡単な実装を始めるため、次を準備すること。

- 開発環境（Python / R / Julia いずれか）  
- 乱数・配列操作・可視化の基本ライブラリ  
- コード再実行可能な構成（Notebook または script + README）  
- 目的関数を「候補を受け取り，評価値を返す関数」として書く習慣

Pythonを選ぶ場合の最小例（参考）:


In [1]:
import random
from math import sin

def objective(x: list[int]) -> int:
    """0/1 ベクトルに対する目的関数の最小例（ダミー）。

    Args:
        x: 0 または 1 からなる整数リスト。

    Returns:
        ``x`` の要素の和。
    """
    return sum(x)

candidate = [random.randint(0, 1) for _ in range(10)]
print(candidate, objective(candidate))

def black_box_score(a: float) -> float:
    """中身を直接使わず，評価値だけを見る関数の例。"""
    return (a - 1.5) ** 2 + 0.2 * sin(5.0 * a)

trial = random.uniform(-3.0, 3.0)
print(trial, black_box_score(trial))


[0, 0, 1, 0, 0, 0, 1, 1, 1, 1] 5
2.244890173310413 0.3600769912400327


```{dropdown} NOTE: サンプルコードの位置づけ
:class: dropdown
:open: true
上記コードは「最適化アルゴリズム」ではなく、目的関数評価の最小例である。  
第2回以降では，このような関数を何度も評価しながら，より良い候補を探すアルゴリズムを追加する。
```

---


## 8. まとめ

- 組合せ最適化は、離散的な解候補から制約を満たす最良解を探す問題である。  
- ブラックボックス最適化は，目的関数の中身を直接使わず，候補を評価して得られる値を頼りに良い解を探す問題である。  
- 実問題では組合せ爆発や評価回数の制約があるため、全探索や無計画な試行は多くの場合非現実的である。  
- そのため厳密解法，近似解法，メタヒューリスティクスを問題の型に応じて使い分ける。  
- 本授業では Binary GA，GA，PSO，ACO，DE，CMA-ES を通じて，候補解の表現と探索の設計を学ぶ。  
- 第2回以降は、モデル化の精度と実装力の両方を段階的に強化する。

---


## 9. 確認問題

1. 最適化問題における「解」「目的関数」「制約」をそれぞれ説明せよ。  
2. 組合せ最適化とブラックボックス最適化の違いを，解の表現と目的関数の扱いの観点から述べよ。  
3. 20都市のTSPで全探索が難しい理由を、解候補数の観点から述べよ。  
4. ハイパーパラメータ探索がブラックボックス最適化として扱われる理由を述べよ。  
5. 自分が興味を持つ実問題を1つ挙げ、解表現と評価方法の案を示せ。

```{admonition} 次回予告
:class: hint dropdown
:open: true
第2回は「バイナリ遺伝的アルゴリズム（Binary Genetic Algorithm, Binary GA）」を扱う。  
0/1 ベクトルで解を表し，特徴を使う／使わないという選択問題を，選択・交叉・突然変異・世代交代の流れで探索する。
```
